# Bible Alignment Processor
Converts `bible/<book>/<chapter>.txt` → `bible/<book>/<chapter>.json`  
Failed batches are saved to `failures/<book>/<chapter>_batch<N>.txt` with full raw model output for inspection.

In [ ]:
# ── Configuration — edit these ────────────────────────────────────────────────

MODEL            = "qwen3.5:122b"
OLLAMA_URL       = "http://localhost:11434/api/chat"
BIBLE_ROOT       = "bible"          # folder containing book subfolders
FAILURES_ROOT    = "failures"       # where failed raw outputs land
VERSES_PER_BATCH = 6                # lower to 3 if JSON parse errors persist
REQUEST_TIMEOUT  = 600              # seconds per Ollama call
MAX_RETRIES      = 3
TEMPERATURE      = 0.1
NUM_CTX          = 18384  # context window: input + output combined
NUM_PREDICT      = 10192   # max output tokens — 6 verbose verses can be ~4000 tokens

# Set to a list of paths to process only specific files, e.g.:
# ONLY_FILES = ["bible/1_samuel/1.txt"]
# Set to None to process all .txt files under BIBLE_ROOT
ONLY_FILES       = None

# Set to True to retry files where a previous run wrote an _error JSON
RETRY_FAILED     = True

print("Configuration loaded.")

In [ ]:
import json
import re
import time
import textwrap
from pathlib import Path
import requests
print("Imports OK")

In [ ]:
SYSTEM_PROMPT = """/no_think
From now on in this chat do this task i will describe. Your task is to align bible translations. I will give you the same verses in hebrew, then greek, then latvian. For each Hebrew word match zero or more Greek and zero or more Latvian words that are the translations. This is from Genesis. The need is for accuracy and completeness in the mapping, even when word orders don't align perfectly. I am studying translation choices how concepts are rendered or lost from originals to translations and considering possible parallel concordance build. Do not modify hebrew word order or words in any way.
input format: Continous block of all the verses intermixing the translations. First hebrew, then after newline greek then after newline latvian; after newline next verse in the three languages. Each greek and latvian line is already aligned with corresponding hebrew line, but word order is not. Each word might have zero or more words in other language or vice versa. To get the hebrew words from sentence, split hebrew on spaces. No need to group or ungroup the punctation even where i assume פ is punctation not part of word by seperating it using space from the other part of word. In such cases feel safe to output that פ as another word, which, of course, will have empty latvian and greek mapping lists for that word. If hypens or other punctation like ׃ are in the resulting space split word, feel safe to treat that as an unit for the output as later I will check whether splits are the same as words you specified in output.
output: Now I want word level mapping. Return only valid JSON. No need to pretty print. No explanations. No comments. Preserve hebrew word order. Do not default to outputing empty lists and putting all the words in leftover, but use leftover only when there seems to be extra words that do not fit to any of the original Hebrew words. Create JSON for all the verses, putting each verse into a seperate list member. Try not to loose hebrew words. If Latvian words in the verse remain unmapped, include them in leftover_latvian list. If greek remain unmapped, include the in leftover_greek list.
Example of input and resulting output:
output format sample from Genesis chapter 9 verses 28 and 29:
data:

וַֽיְחִי־ נֹ֖חַ אַחַ֣ר הַמַּבּ֑וּל שְׁלֹ֤שׁ מֵאוֹת֙ שָׁנָ֔ה וַֽחֲמִשִּׁ֖ים שָׁנָֽה׃
ἔζησεν δὲ Νωε μετὰ τὸν κατακλυσμὸν τριακόσια πεντήκοντα ἔτη
Un Noa dzīvoja pēc ūdensplūdiem trīs simti piecdesmit gadus
וַיִּֽהְיוּ֙ כָּל־ יְמֵי־ נֹ֔חַ תְּשַׁ֤ע מֵאוֹת֙ שָׁנָ֔ה וַחֲמִשִּׁ֖ים שָׁנָ֑ה וַיָּמֹֽת׃ פ
καὶ ἐγένοντο πᾶσαι αἱ ἡμέραι Νωε ἐννακόσια πεντήκοντα ἔτη καὶ ἀπέθανεν
Un visi Noas mūža gadi bija deviņi simti piecdesmit tad viņš nomira
output:
[
{
    "hebrew_words": [
      {"hebrew": "וַֽיְחִי־", "greek": ["ἔζησεν", "δὲ"], "latvian": ["Un", "dzīvoja"] },
      {"hebrew": "נֹ֖חַ", "greek": ["Νωε"], "latvian": ["Noa"] },
      {"hebrew": "אַחַ֣ר", "greek": ["μετὰ"], "latvian": ["pēc"] },
      {"hebrew": "הַמַּבּ֑וּל", "greek": ["τὸν", "κατακλυσμόν"], "latvian": ["ūdensplūdiem"] },
      {"hebrew": "שְׁלֹ֤שׁ", "greek": ["τριακόσια"], "latvian": ["trīs", "simti"] },
      {"hebrew": "מֵאוֹת֙", "greek": [], "latvian": [] },
      {"hebrew": "שָׁנָ֔ה", "greek": ["ἔτη"], "latvian": ["gadus"] },
      {"hebrew": "וַֽחֲמִשִּׁ֖ים", "greek": ["πεντήκοντα"], "latvian": ["piecdesmit"] },
      {"hebrew": "שָׁנָֽה׃", "greek": [], "latvian": [] }
    ],
    "leftover_greek": [],
    "leftover_latvian": []
  },
{
    "hebrew_words": [
      {"hebrew": "וַיִּֽהְיוּ֙", "greek": ["καὶ", "ἐγένοντο"], "latvian": ["Un"] },
      {"hebrew": "כָּל־", "greek": ["πᾶσαι"], "latvian": ["visi"] },
      {"hebrew": "יְמֵי־", "greek": ["αἱ", "ἡμέραι"], "latvian": ["mūža"] },
      {"hebrew": "נֹ֔חַ", "greek": ["Νωε"], "latvian": ["Noas"] },
      {"hebrew": "תְּשַׁ֤ע", "greek": ["ἐννακόσια"], "latvian": ["deviņi", "simti"] },
      {"hebrew": "מֵאוֹת֙", "greek": [], "latvian": [] },
      {"hebrew": "שָׁנָ֔ה", "greek": ["πεντήκοντα"], "latvian": ["piecdesmit"] },
      {"hebrew": "וַחֲמִשִּׁ֖ים", "greek": [], "latvian": [] },
      {"hebrew": "שָׁנָ֑ה", "greek": ["ἔτη"], "latvian": ["gadi"] },
      {"hebrew": "וַיָּמֹֽת׃", "greek": ["καὶ", "ἀπέθανεν"], "latvian": ["tad", "viņš", "nomira"] }
    ],
    "leftover_greek": [],
    "leftover_latvian": ["bija"]
  }
]

now the task data:
"""
print("System prompt loaded.")

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

def parse_verses(text: str) -> list:
    """Parse .txt into list of (hebrew, greek, latvian) triples."""
    lines = [l.rstrip() for l in text.splitlines()]
    non_empty = [l for l in lines if l.strip()]
    if len(non_empty) % 3 != 0:
        raise ValueError(
            f"Line count {len(non_empty)} is not divisible by 3. "
            "Each verse needs exactly 3 lines: hebrew, greek, latvian."
        )
    return [(non_empty[i], non_empty[i+1], non_empty[i+2])
            for i in range(0, len(non_empty), 3)]


def verses_to_prompt(verses: list) -> str:
    return "\n".join(f"{h}\n{g}\n{l}" for h, g, l in verses)


def call_ollama(user_content: str) -> str:
    payload = {
        "model": MODEL,
        "stream": False,
        "think": False,          # Ollama native flag — disables Qwen3/3.5 thinking mode
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_content},
        ],
        "options": {"temperature": TEMPERATURE, "num_ctx": NUM_CTX, "num_predict": NUM_PREDICT},
    }
    resp = requests.post(OLLAMA_URL, json=payload, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    content = resp.json()["message"]["content"]
    # Strip any residual <think>...</think> blocks the model emitted anyway
    content = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()
    return content


def extract_json(raw: str) -> list:
    cleaned = re.sub(r"```json\s*", "", raw)
    cleaned = re.sub(r"```\s*", "", cleaned).strip()
    try:
        result = json.loads(cleaned)
        if isinstance(result, list):
            return result
    except json.JSONDecodeError:
        pass
    start = cleaned.find("[")
    if start == -1:
        raise ValueError("No JSON array found in model output")
    depth, end = 0, -1
    for i, ch in enumerate(cleaned[start:], start):
        if ch == "[": depth += 1
        elif ch == "]":
            depth -= 1
            if depth == 0:
                end = i
                break
    if end == -1:
        raise ValueError("Unmatched '[' — response likely truncated")
    result = json.loads(cleaned[start:end+1])
    if not isinstance(result, list):
        raise ValueError("Extracted JSON is not a list")
    return result


def is_error_json(path: Path) -> bool:
    """Return True if path contains a sentinel _error JSON from a previous failed run."""
    try:
        d = json.loads(path.read_text(encoding="utf-8"))
        return isinstance(d, dict) and d.get("_error", False)
    except Exception:
        return True  # corrupt JSON → treat as failed


print("Helpers defined.")


In [ ]:
# ── Collect files to process ──────────────────────────────────────────────────

if ONLY_FILES:
    txt_files = [Path(f) for f in ONLY_FILES]
else:
    txt_files = sorted(Path(BIBLE_ROOT).glob("**/*.txt"))

# Filter out already-completed files (unless they are error sentinels)
to_process = []
to_skip    = []
for f in txt_files:
    jp = f.with_suffix(".json")
    if jp.exists():
        if is_error_json(jp) and RETRY_FAILED:
            to_process.append(f)
        else:
            to_skip.append(f)
    else:
        to_process.append(f)

print(f"Found {len(txt_files)} .txt files total")
print(f"  Will process : {len(to_process)}")
print(f"  Will skip    : {len(to_skip)} (valid JSON already exists)")
print()
for f in to_process:
    print(f"  → {f}")

In [ ]:
# ── Main processing loop ──────────────────────────────────────────────────────
# Runs through every file, prints raw model output immediately on any failure
# so you can tune the prompt without waiting for the full run to finish.

failures_root = Path(FAILURES_ROOT)

stats = {"ok": 0, "failed": 0, "skipped": len(to_skip)}

for txt_path in to_process:
    json_path = txt_path.with_suffix(".json")
    rel = txt_path.relative_to(BIBLE_ROOT)
    print(f"\n{'='*60}")
    print(f"FILE: {txt_path}")
    print(f"{'='*60}")

    # Parse file
    try:
        text   = txt_path.read_text(encoding="utf-8")
        verses = parse_verses(text)
        print(f"  Parsed {len(verses)} verses, "
              f"{(len(verses) + VERSES_PER_BATCH - 1) // VERSES_PER_BATCH} batches of {VERSES_PER_BATCH}")
    except Exception as e:
        print(f"  ✗ PARSE ERROR: {e}")
        stats["failed"] += 1
        continue

    all_results = []
    file_ok     = True

    for batch_start in range(0, len(verses), VERSES_PER_BATCH):
        batch     = verses[batch_start : batch_start + VERSES_PER_BATCH]
        batch_num = batch_start // VERSES_PER_BATCH + 1
        prompt    = verses_to_prompt(batch)
        v_range   = f"verses {batch_start+1}–{batch_start+len(batch)}"
        raw       = None

        for attempt in range(1, MAX_RETRIES + 1):
            print(f"  batch {batch_num} ({v_range}) attempt {attempt}/{MAX_RETRIES} … ",
                  end="", flush=True)
            try:
                t0  = time.time()
                raw = call_ollama(prompt)
                elapsed = time.time() - t0
                parsed  = extract_json(raw)

                if len(parsed) != len(batch):
                    print(f"⚠ got {len(parsed)} objects for {len(batch)} verses ({elapsed:.1f}s) — accepting")
                else:
                    print(f"✓ ({elapsed:.1f}s)")

                all_results.extend(parsed)
                break  # success

            except requests.exceptions.Timeout:
                print(f"✗ TIMEOUT after {REQUEST_TIMEOUT}s")
                if attempt == MAX_RETRIES:
                    file_ok = False
                else:
                    time.sleep(5)

            except requests.exceptions.RequestException as e:
                print(f"✗ HTTP ERROR: {e}")
                if attempt == MAX_RETRIES:
                    file_ok = False
                else:
                    time.sleep(5)

            except (ValueError, json.JSONDecodeError) as e:
                print(f"✗ JSON PARSE ERROR: {e}")

                # ── Show the raw output immediately ───────────────────────────
                if raw is not None:
                    # print()
                    # print("  ┌─ RAW MODEL OUTPUT (first 2000 chars) ─────────────────")
                    # preview = raw[:2000]
                    # for line in preview.splitlines():
                    #     print(f"  │ {line}")
                    # if len(raw) > 2000:
                    #     print(f"  │ … ({len(raw) - 2000} more chars)")
                    # print("  └────────────────────────────────────────────────────────")
                    # print()

                    # Save full raw output to failures folder
                    fail_dir = failures_root / rel.parent
                    fail_dir.mkdir(parents=True, exist_ok=True)
                    fail_path = fail_dir / f"{txt_path.stem}_batch{batch_num}_attempt{attempt}.txt"
                    fail_path.write_text(
                        f"=== PROMPT ===\n{prompt}\n\n=== RAW MODEL OUTPUT ===\n{raw}",
                        encoding="utf-8"
                    )
                    print(f"  Full output saved → {fail_path}")
                    print()

                if attempt == MAX_RETRIES:
                    file_ok = False
                else:
                    time.sleep(2)

        if not file_ok:
            break  # don't process remaining batches after a hard failure

    if file_ok:
        json_path.write_text(
            json.dumps(all_results, ensure_ascii=False, indent=1),
            encoding="utf-8"
        )
        print(f"  ✓ Wrote {json_path} ({len(all_results)} verses)")
        stats["ok"] += 1
    else:
        # Write error sentinel *next to* <-- write_text only overwrites... <-- source so RETRY_FAILED can find it
        error_obj = {
            "_error": True,
            "file": str(txt_path),
            "partial_verses_saved": len(all_results),
        }
        with json_path.open("a") as f:
            f.write(json.dumps(error_obj, ensure_ascii=False, indent=3),
                                 encoding="utf-8")
            f.write(
                json.dumps(all_results, ensure_ascii=False, indent=1),
                encoding="utf-8"
            )
        print(f"  ✗ FAILED — error sentinel written to {json_path}")
        print(f"    Check failures/{rel.parent}/ for raw model outputs")
        stats["failed"] += 1

print()
print("="*60)
print(f"DONE  ✓ OK={stats['ok']}  ✗ Failed={stats['failed']}  — Skipped={stats['skipped']}")
print("="*60)

In [ ]:
#for i in range (2, 10):
#    !rm bible/1_samuel/{i}.json
#!rm bible/1_samuel/20.txt
#!rm bible/1_samuel/20.json

In [ ]:
# ── Inspect failures ──────────────────────────────────────────────────────────
# Run this cell any time to list everything in the failures/ folder

fail_files = sorted(Path(FAILURES_ROOT).glob("**/*.txt")) if Path(FAILURES_ROOT).exists() else []

if not fail_files:
    print("No failure files found — all good!")
else:
    print(f"{len(fail_files)} failure file(s):")
    for f in fail_files:
        print(f"  {f}")

In [ ]:
# ── Read a specific failure file ──────────────────────────────────────────────
# Edit the path below and run this cell to inspect what the model actually returned

INSPECT_FILE = ""  # e.g. "failures/1_samuel/1_batch1_attempt3.txt"

if INSPECT_FILE:
    content = Path(INSPECT_FILE).read_text(encoding="utf-8")
    # Split into prompt and output sections
    if "=== RAW MODEL OUTPUT ===" in content:
        prompt_part, output_part = content.split("=== RAW MODEL OUTPUT ===", 1)
        print("── PROMPT ────────────────────────────────────────────────")
        print(prompt_part.replace("=== PROMPT ===", "").strip())
        print()
        print("── RAW MODEL OUTPUT ──────────────────────────────────────")
        print(output_part.strip())
    else:
        print(content)
else:
    print("Set INSPECT_FILE to a path from the cell above to view it here.")